In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import re

def extrair_unidades(texto):
    """Extrai quantidade de UND no texto (retorna int ou 1 se não achar)."""
    if pd.isna(texto):
        return 1
    texto = str(texto).lower()
    m = re.search(r'(\d+)\s*und', texto)
    if m:
        return int(m.group(1))
    return 1

def ajustar_por_material(df):
    """Corrige previsões IA usando MATERIAL como dominante."""
    for i, row in df.iterrows():
        qtd_mat = extrair_unidades(row['MATERIAL'])
        qtd_conc = extrair_unidades(row['NOME_CONC'])

        if qtd_mat != qtd_conc:
            if qtd_mat == 1 and qtd_conc > 1:
                df.at[i, 'CONVERSAO_PREV'] = 'divide'
                df.at[i, 'FATOR_PREV'] = qtd_conc
            elif qtd_mat > 1 and qtd_conc == 1:
                df.at[i, 'CONVERSAO_PREV'] = 'multiplica'
                df.at[i, 'FATOR_PREV'] = qtd_mat
    return df


# -------------------------
# Treino
# -------------------------
def treinar_modelos(caminho_treino):
    df = pd.read_excel(caminho_treino)

    # garantir colunas
    for col in ['MATERIAL', 'NOME_CONC', 'CONVERSAO', 'FATOR']:
        if col not in df.columns:
            raise KeyError(f"Coluna {col} não encontrada.")

    df['MATERIAL'] = df['MATERIAL'].astype(str).fillna('')
    df['NOME_CONC'] = df['NOME_CONC'].astype(str).fillna('')

    # features de texto
    df['texto'] = df['MATERIAL'] + " | " + df['NOME_CONC']

    X = df[['texto']]
    y_conv = df['CONVERSAO'].astype(str)
    y_fator = pd.to_numeric(df['FATOR'], errors='coerce').fillna(0.0)

    # pré-processador (TF-IDF sobre texto)
    preproc = ColumnTransformer([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=20000), 'texto')
    ])

    # classificador para conversão
    modelo_conv = Pipeline([
        ('pre', preproc),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ])

    # regressor para fator
    modelo_fator = Pipeline([
        ('pre', preproc),
        ('reg', RandomForestRegressor(n_estimators=200, random_state=42))
    ])

    # treino
    modelo_conv.fit(X, y_conv)
    modelo_fator.fit(X, y_fator)

    # avaliação simples
    X_tr, X_te, y_tr, y_te, f_tr, f_te = train_test_split(X, y_conv, y_fator, test_size=0.1, random_state=42)
    print("Classificação (CONVERSAO):")
    print(classification_report(y_te, modelo_conv.predict(X_te)))
    print("Erro Médio Absoluto (FATOR):", mean_absolute_error(f_te, modelo_fator.predict(X_te)))

    return modelo_conv, modelo_fator

# -------------------------
# Predição em novos dados
# -------------------------
def prever(modelo_conv, modelo_fator, caminho_novos, caminho_saida=None):
    df = pd.read_excel(caminho_novos)

    df['MATERIAL'] = df['MATERIAL'].astype(str).fillna('')
    df['NOME_CONC'] = df['NOME_CONC'].astype(str).fillna('')
    df['texto'] = df['MATERIAL'] + " | " + df['NOME_CONC']

    X_new = df[['texto']]

    df['CONVERSAO_PREV'] = modelo_conv.predict(X_new)
    df['FATOR_PREV'] = modelo_fator.predict(X_new).round(4)  # arredondar p/ evitar quebrados feios

    if caminho_saida:
        df.to_excel(caminho_saida, index=False)
        print(f"Resultado salvo em: {caminho_saida}")

    return df

# -------------------------
# Uso
# -------------------------
if __name__ == "__main__":
    caminho_treino = r"C:\Users\SeuUsuario\Downloads\Banco_de_Fardos.xlsx"
    caminho_novos = r"C:\Users\SeuUsuario\Downloads\novos_fardos.xlsx"
    saida = r"C:\Users\SeuUsuario\Downloads\Resultado_Prev.xlsx"

    modelo_conv, modelo_fator = treinar_modelos(caminho_treino)
    resultado = prever(modelo_conv, modelo_fator, caminho_novos, caminho_saida=saida)

    print(resultado.head(20))
